# RABBiT demo — speech to fMRI in six sections

This notebook walks through everything you can do with the RABBiT package today:

1. **Load** a pretrained checkpoint in two lines.
2. **Predict** fMRI responses for a single audio clip.
3. **Visualize** the prediction with the audio waveform on top — see, side by side, what the model thinks the brain is doing while it hears the stimulus.
4. **Batch-predict** over a directory of audio clips.
5. **Evaluate** the model on held-out narratives subjects and inspect per-ROI correlation.
6. **Compare hemispheres** for one ROI.

The inference function accepts `audio_onset`, `shift`, `tr_length`, `hrf_delay`, etc. as keyword arguments with paper-sensible defaults — see section 2 below for what each one does.

Edit the paths at the top of section 0 to match your filesystem layout.

## 0 — Setup

Imports + paths + a device sanity-check. The package is in `src/` (editable install) so `PYTHONPATH=src` is enough when launching the kernel.

In [ ]:
import os, sys, json
from pathlib import Path

# Make sure src/ is importable when the notebook lives under notebooks/.
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt

import rabbit
from rabbit.inference import ROIPredictor
from rabbit.viz import (
    plot_audio_and_responses, plot_roi_grid, plot_narratives_summary,
    audio_envelope,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'rabbit {rabbit.__version__}  ·  device={device}  ·  torch={torch.__version__}')

# ---- EDIT these paths to match your setup ---------------------------------
# Grab a checkpoint from the Hugging Face model repo (omermosa/rabbit).
CHECKPOINT = 'path/to/rabbit_friends_fs6_shared_dev.pt'
CONFIG     = REPO_ROOT / 'configs' / 'friends_shared_dev.yaml'

# Demo audio (single clip): a 16 kHz mono waveform saved as .npy.
DEMO_AUDIO_NPY = Path('path/to/clip.npy')

# Directory of short audio snippets (.wav) for batch inference.
DEMO_AUDIO_DIR = Path('path/to/audio_snippets')

# Narratives held-out story (e.g. "21st Year"): 16 kHz audio, the .report file
# with TR triggers, and one fMRI .npy per held-out subject.
NARR_AUDIO    = Path('path/to/narratives/21styear_audio_16k.npy')
NARR_REPORT   = Path('path/to/narratives/21styear.report')
NARR_FMRI_DIR = Path('path/to/narratives/fmri')
NARR_SUBJECTS = [244, 249, 254, 255, 256, 257, 258, 259, 260, 261,
                 262, 263, 264, 265, 266, 267, 268, 269, 270, 271]

print(f'\nCHECKPOINT  exists: {Path(CHECKPOINT).exists()}')
print(f'CONFIG      exists: {CONFIG.exists()}')
print(f'DEMO_AUDIO  exists: {DEMO_AUDIO_NPY.exists()}')
print(f'NARR_AUDIO  exists: {NARR_AUDIO.exists()}')

## 1 — Load the model

`ROIPredictor.from_checkpoint` builds the model from a YAML config + checkpoint file, applies the avg-dev trick by default (so the model is ready for held-out-subject inference), and returns a thin wrapper around the model.

In [ ]:
predictor = ROIPredictor.from_checkpoint(
    checkpoint_path=CHECKPOINT,
    config_path=CONFIG,
    device=device,
    use_avg_dev=True,   # average deviation bases across training subjects (zero-shot mode)
)

m = predictor.model
print(f'readout_type:  {m.readout_type}')
print(f'hidden_dim:    {m.hidden_dim}')
print(f'output_dim:    {m.output_dim}      ← fs6 30-ROI flat vertex count')
print(f'n_subjects:    {next(iter(m.roi_readouts.values())).n_subjects}')
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'trainable params (after load): {n_params:,}')

## 2 — Predict on a single audio clip

The simple API. `predictor.predict()` accepts an audio path, an `np.ndarray`, or a `torch.Tensor` at 16 kHz, and runs the full inference pipeline:

- splits audio into per-TR windows of length `tr_length` seconds,
- stacks `range(0, hrf_delay+1)` shifted copies along the feature axis,
- trims `trim_start` / `trim_end` edge TRs,
- forwards through the model TR-by-TR (batched on GPU).

All keyword arguments have paper-sensible defaults:

| arg | default | meaning |
|---|---|---|
| `tr_length`   | `1.49` | TR period in seconds (Friends; use `2.0` for moth) |
| `hrf_delay`   | `6`    | number of HRF-lag TRs |
| `trim_start`  | `10`   | leading TRs to drop |
| `trim_end`    | `9`    | trailing TRs to drop |
| `audio_onset` | `0.0`  | seconds of leading silence before the stimulus |
| `shift`       | `0`    | integer TR offset applied to the returned `tr_times` only |
| `batch_size`  | `32`   | TRs per GPU forward pass |

For the demo we'll grab the first 60 seconds of a Friends episode.

In [ ]:
DEMO_SECONDS = 60
wav_full = np.load(DEMO_AUDIO_NPY).astype(np.float32).squeeze()
demo_audio = wav_full[: DEMO_SECONDS * 16_000]
print(f'demo audio: {demo_audio.shape[0] / 16_000:.1f}s @ 16 kHz  ·  rms={np.sqrt((demo_audio**2).mean()):.3f}')

# One-line inference. All other args use their defaults.
result = predictor.predict(
    demo_audio,
    tr_length=1.49,
    hrf_delay=6,
    trim_start=2,            # smaller edge trim — we only have 60 s of audio
    trim_end=2,
    audio_onset=0.0,
)

print(f'\nresult.fmri          shape: {result.fmri.shape}    ← (n_TRs, 41394)')
print(f'result.tr_times[:3]:        {result.tr_times[:3]}')
print(f'result.tr_times[-3:]:       {result.tr_times[-3:]}')
print(f'finite fraction:            {np.isfinite(result.fmri).mean():.4f}')
print(f'\nresult.by_roi["aud_primary_lh"] shape: {result.by_roi["aud_primary_lh"].shape}')

## 3 — Visualize: audio waveform + predicted responses

`plot_audio_and_responses` puts the audio envelope on the top panel and the model's predicted ROI-mean response on each subsequent panel, all time-aligned. By default we show the canonical auditory→language hierarchy `A1 → Belt → STG → IFG`.

Watch the A1 trace track the audio envelope — that's the model's primary-auditory response correctly following acoustic energy. By contrast IFG (Broca-adjacent) responds with longer-timescale dynamics, less tightly locked to the moment-to-moment audio.

In [ ]:
fig, axes = plot_audio_and_responses(
    audio=demo_audio,
    result=result,
    sample_rate=16_000,
    audio_onset=0.0,
    roi_bases=('aud_primary', 'aud_belt', 'stg_sts', 'ifg'),
    title='RABBiT prediction on a 60-second Friends clip',
)
plt.show()

### Zooming in

Pass a `time_window=(lo, hi)` to focus on a single 20-second window. This makes the temporal coupling between audio and A1 easier to see.

In [ ]:
fig, axes = plot_audio_and_responses(
    audio=demo_audio,
    result=result,
    roi_bases=('aud_primary', 'aud_belt', 'stg_sts', 'ifg'),
    time_window=(15.0, 35.0),
    title='Zoom: 15 to 35 s',
)
plt.show()

### Small-multiples ROI grid

`plot_roi_grid` shows traces for all nine language ROIs at once. Useful for spotting which ROIs respond similarly and which ones diverge.

In [ ]:
fig, axes = plot_roi_grid(result, ncols=3, title='Per-ROI mean predicted response (60-s Friends clip)')
plt.show()

## 4 — Batch inference over a directory

`predict_many` accepts either a directory path (it lists audio files matching `.wav|.flac|.npy|.mp3`) or an explicit list of file paths, and returns an `OrderedDict[name → RABBiTPrediction]`. Each file uses the same prediction kwargs you pass in.

In [ ]:
# Use the first six 10-second snippets so the run stays quick.
snippet_paths = sorted(DEMO_AUDIO_DIR.glob('snippet_*.wav'))[:6]
print(f'predicting on {len(snippet_paths)} clips ...')

batch_results = predictor.predict_many(
    snippet_paths,
    tr_length=1.49,
    hrf_delay=6,
    trim_start=2, trim_end=1,
    audio_onset=0.0,
    progress=True,
)

print('\nDone. Predicted clips:')
for name, r in batch_results.items():
    print(f'  {name:<14} n_TRs={r.fmri.shape[0]:>2}  duration~{r.tr_times[-1]:.1f}s')

In [ ]:
# Plot one clip alongside its prediction — same plot as section 3, different audio.
demo_name = list(batch_results.keys())[0]
import soundfile as sf
wav_clip, _ = sf.read(snippet_paths[0])
fig, axes = plot_audio_and_responses(
    audio=wav_clip,
    result=batch_results[demo_name],
    roi_bases=('aud_primary', 'stg_sts', 'ifg'),
    title=f'Snippet: {demo_name}',
)
plt.show()

## 5 — Evaluate on held-out narratives subjects

`evaluate_on_narratives` runs the full zero-shot eval pipeline: audio → TR alignment → forward → shift sweep → 4-fold z-scored Pearson correlation against the 20-subject group-mean fMRI on `fsaverage6`. Produces a per-vertex correlation tensor (41,394,) and per-ROI summary.

First time this runs, `mne` parses the HCP-MMP1 annotation on `fsaverage6` (takes a few seconds and prints some log lines).

In [ ]:
from rabbit.eval import NarrativesStory, evaluate_on_narratives

story = NarrativesStory(
    name='21styear',
    audio_path=NARR_AUDIO,
    report_path=NARR_REPORT,
    fmri_paths=[NARR_FMRI_DIR / f'sub_{s}.npy' for s in NARR_SUBJECTS],
)

result_eval = evaluate_on_narratives(
    model=predictor.model,
    story=story,
    tr_length=1.49,
    hrf_delay=6,
    trim_start=10,
    trim_end=9,
    batch_size=16,
    device=device,
)

print(f'applied shift:        {result_eval.applied_shift}  ({result_eval.applied_shift * 1.49:.2f}s)')
print(f'mean vertex_corr:     {result_eval.vertex_corr.mean():+.4f}')
print(f'median vertex_corr:   {np.median(result_eval.vertex_corr):+.4f}')
print('\nshift sweep:')
for sh, mr in result_eval.shift_sweep.items():
    marker = '  ← picked' if sh == result_eval.applied_shift else ''
    print(f'  shift {sh:+d} ({sh*1.49:+.2f}s):  mean_r={mr:+.4f}{marker}')

In [ ]:
fig, ax = plot_narratives_summary(
    result_eval, sort=True,
    title='RABBiT zero-shot on 21st Year — per-ROI Pearson r (4-fold z-scored)',
)
plt.show()

# A quick ranked view of the per-ROI table, in case the bar chart is too dense.
print('\nTop 8 ROIs by mean r:')
for name, r in sorted(result_eval.roi_corrs.items(), key=lambda kv: -kv[1])[:8]:
    print(f'  {name:<22}  {r:+.4f}')

## 6 — Compare hemispheres for one ROI

The model's flat output is in alternating LH/RH order. `result.by_roi[name]` gives the per-vertex predictions for one hemisphere of one ROI. We can compare LH vs RH for a single ROI as a sanity check on bilateral symmetry.

In [ ]:
with plt.rc_context({'font.family': 'serif', 'mathtext.fontset': 'stix'}):
    fig, ax = plt.subplots(figsize=(10, 3.0))
    times = result.tr_times
    ax.plot(times, result.by_roi['aud_primary_lh'].mean(1), color='#882255', label='A1 (left)', linewidth=1.6)
    ax.plot(times, result.by_roi['aud_primary_rh'].mean(1), color='#88CCEE', label='A1 (right)', linewidth=1.6, linestyle='--')
    ax.axhline(0, color='0.7', linewidth=0.5)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('mean predicted response')
    ax.set_title('A1 — left vs right hemisphere (60-s Friends clip)')
    ax.legend(loc='upper right', frameon=False)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.tight_layout()
plt.show()

lh = result.by_roi['aud_primary_lh'].mean(1)
rh = result.by_roi['aud_primary_rh'].mean(1)
print(f'\nPearson(LH, RH) for A1 mean trace: {np.corrcoef(lh, rh)[0, 1]:+.4f}')

---

## Things to try next

- Swap `CHECKPOINT` to a `friends_shared_dev_wavlm` checkpoint and re-run section 3 — WavLM was the headline winner; you should see a slightly tighter A1 envelope-tracking.
- Pass `return_attention=True` to `predictor.predict(...)` and inspect `result.attention` — shape `(n_TRs, n_heads=8, n_rois=30, n_audio_tokens)`. Each head specializes on a different acoustic timescale.
- Use `evaluate_on_narratives(..., shift=0)` to force a non-shifted comparison instead of the auto-picked best shift, matching the LPP-protocol convention.
- Run `predict_many` on a directory of your own audio and overlay the resulting traces.

For the model internals and the shared+deviation math, see [`docs/architecture.md`](../docs/architecture.md). For training a new checkpoint, see [`docs/training.md`](../docs/training.md).